# Stacking de modelos

Implementamos un enfoque de stacking con el objetivo de combinar modelos cuyos errores sean lo más diversos posible. Esto resulta especialmente adecuado en un problema de clasificación multiclase evaluado mediante Kappa, ya que esta métrica penaliza fuertemente los errores sistemáticos entre clases.

Para esto, seleccionamos modelos de distintas familias (boosting y bagging), buscando que cada uno capture patrones diferentes del dataset. La combinación de estas predicciones a través de un meta-modelo permite mejorar la capacidad de generalización respecto a modelos individuales.

Los modelos base a implementar son:

LightGBM (Boosting)	=> Construye árboles secuencialmente, cada uno corrige los errores del anterior

CatBoost (Boosting)	=>  Boosting especializado en variables categóricas

Random Forest (Bagging) =>	Construye muchos árboles en paralelo sobre muestras aleatorias y promedia

ExtraTreesClassifier (Bagging) => Como Random Forest pero con splits aún más aleatorios (más rápido, más diverso)



Imports de las librerias

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression

import optuna

Carga de datos

In [2]:
BASE_DIR = '..'
TARGET = 'AdoptionSpeed'
SEED = 42
N_FOLDS = 5
N_CLASSES = 5

train = pd.read_csv(os.path.join(BASE_DIR, 'work/cleaned/train_X2.csv'))
test  = pd.read_csv(os.path.join(BASE_DIR, 'work/cleaned/test_X2.csv'))

X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]
y_test  = test[TARGET] if TARGET in test.columns else None
X_test  = test.drop(columns=[TARGET], errors="ignore")

print(f"Train: {X_train.shape} | Test: {X_test.shape}")


Train: (11994, 36) | Test: (2999, 36)


## Nested CV (CV anidado)

Para evitar la contaminación entre la optimización de hiperparámetros y la generación de predicciones OOF, implementamos **Nested CV**:

- **Loop externo** (5 folds): define los folds OOF para el stacking.
- **Loop interno** (3 folds + Optuna, 15 trials): para cada fold externo, optimiza los HPs usando **únicamente** los datos de entrenamiento de ese fold.

De esta forma, los HPs de cada modelo se ajustan al mismo subconjunto de datos sobre el cual el modelo luego predice en OOF, eliminando el sesgo optimista que existía antes al optimizar con todo el dataset.

El costo es mayor (5 × 4 modelos × 15 trials × 3 folds internos), pero la estimación de performance es confiable.

In [3]:
from lightgbm import early_stopping as lgbm_es
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

OUTER_FOLDS    = 5
INNER_FOLDS    = 3
N_INNER_TRIALS = 15

outer_skf = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=SEED)
inner_skf = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=SEED)

model_names = ['lgbm', 'catboost', 'rf', 'extratrees']
oof_preds  = {name: np.zeros((len(X_train), N_CLASSES)) for name in model_names}
test_preds = {name: np.zeros((len(X_test),  N_CLASSES)) for name in model_names}

for fold, (outer_tr_idx, outer_val_idx) in enumerate(outer_skf.split(X_train, y_train)):
    print(f"\n=== Fold externo {fold+1}/{OUTER_FOLDS} ===")

    X_ftr  = X_train.values[outer_tr_idx]
    X_fval = X_train.values[outer_val_idx]
    y_ftr  = y_train.iloc[outer_tr_idx].reset_index(drop=True)
    y_fval = y_train.iloc[outer_val_idx]

    # ── LGBM ──────────────────────────────────────────────────────────────
    def objective_lgbm(trial):
        params = {
            'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
            'num_leaves':        trial.suggest_int('num_leaves', 20, 150),
            'max_depth':         trial.suggest_int('max_depth', 4, 12),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 1.0),
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 5.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 5.0, log=True),
            'n_estimators': 500, 'class_weight': 'balanced',
            'random_state': SEED, 'verbosity': -1,
        }
        kappas = []
        for itr_idx, ival_idx in inner_skf.split(X_ftr, y_ftr):
            m = LGBMClassifier(**params)
            m.fit(X_ftr[itr_idx], y_ftr.iloc[itr_idx],
                  eval_set=[(X_ftr[ival_idx], y_ftr.iloc[ival_idx])],
                  callbacks=[lgbm_es(30, verbose=False)])
            kappas.append(cohen_kappa_score(y_ftr.iloc[ival_idx], m.predict(X_ftr[ival_idx]), weights='quadratic'))
        return np.mean(kappas)

    study_lgbm = optuna.create_study(direction='maximize')
    study_lgbm.optimize(objective_lgbm, n_trials=N_INNER_TRIALS)
    best_lgbm = {**study_lgbm.best_params, 'n_estimators': 500,
                 'class_weight': 'balanced', 'random_state': SEED, 'verbosity': -1}
    mdl_lgbm = LGBMClassifier(**best_lgbm)
    mdl_lgbm.fit(X_ftr, y_ftr)
    oof_preds['lgbm'][outer_val_idx] = mdl_lgbm.predict_proba(X_fval)
    test_preds['lgbm'] += mdl_lgbm.predict_proba(X_test.values) / OUTER_FOLDS
    print(f"  lgbm       kappa fold: {cohen_kappa_score(y_fval, mdl_lgbm.predict(X_fval), weights='quadratic'):.4f}")

    # ── CatBoost ──────────────────────────────────────────────────────────
    def objective_catboost(trial):
        params = {
            'iterations':          trial.suggest_int('iterations', 100, 500),
            'learning_rate':       trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'depth':               trial.suggest_int('depth', 3, 10),
            'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg', 0.5, 10.0, log=True),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
            'early_stopping_rounds': 30,
            'random_state': SEED, 'verbose': 0,
        }
        kappas = []
        for itr_idx, ival_idx in inner_skf.split(X_ftr, y_ftr):
            m = CatBoostClassifier(**params)
            m.fit(X_ftr[itr_idx], y_ftr.iloc[itr_idx],
                  eval_set=(X_ftr[ival_idx], y_ftr.iloc[ival_idx]))
            kappas.append(cohen_kappa_score(y_ftr.iloc[ival_idx], m.predict(X_ftr[ival_idx]), weights='quadratic'))
        return np.mean(kappas)

    study_cb = optuna.create_study(direction='maximize')
    study_cb.optimize(objective_catboost, n_trials=N_INNER_TRIALS)
    best_cb = {**study_cb.best_params, 'early_stopping_rounds': 30, 'random_state': SEED, 'verbose': 0}
    mdl_cb = CatBoostClassifier(**best_cb)
    mdl_cb.fit(X_ftr, y_ftr)
    oof_preds['catboost'][outer_val_idx] = mdl_cb.predict_proba(X_fval)
    test_preds['catboost'] += mdl_cb.predict_proba(X_test.values) / OUTER_FOLDS
    print(f"  catboost   kappa fold: {cohen_kappa_score(y_fval, mdl_cb.predict(X_fval), weights='quadratic'):.4f}")

    # ── Random Forest ─────────────────────────────────────────────────────
    def objective_rf(trial):
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 100, 400),
            'max_depth':         trial.suggest_int('max_depth', 5, 20),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2']),
            'class_weight':      trial.suggest_categorical('class_weight', [None, 'balanced']),
            'random_state': SEED, 'n_jobs': -1,
        }
        kappas = []
        for itr_idx, ival_idx in inner_skf.split(X_ftr, y_ftr):
            m = RandomForestClassifier(**params)
            m.fit(X_ftr[itr_idx], y_ftr.iloc[itr_idx])
            kappas.append(cohen_kappa_score(y_ftr.iloc[ival_idx], m.predict(X_ftr[ival_idx]), weights='quadratic'))
        return np.mean(kappas)

    study_rf = optuna.create_study(direction='maximize')
    study_rf.optimize(objective_rf, n_trials=N_INNER_TRIALS)
    best_rf = {**study_rf.best_params, 'random_state': SEED, 'n_jobs': -1}
    mdl_rf = RandomForestClassifier(**best_rf)
    mdl_rf.fit(X_ftr, y_ftr)
    oof_preds['rf'][outer_val_idx] = mdl_rf.predict_proba(X_fval)
    test_preds['rf'] += mdl_rf.predict_proba(X_test.values) / OUTER_FOLDS
    print(f"  rf         kappa fold: {cohen_kappa_score(y_fval, mdl_rf.predict(X_fval), weights='quadratic'):.4f}")

    # ── ExtraTrees ────────────────────────────────────────────────────────
    def objective_et(trial):
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 100, 400),
            'max_depth':         trial.suggest_int('max_depth', 5, 20),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'class_weight':      trial.suggest_categorical('class_weight', [None, 'balanced']),
            'random_state': SEED, 'n_jobs': -1,
        }
        kappas = []
        for itr_idx, ival_idx in inner_skf.split(X_ftr, y_ftr):
            m = ExtraTreesClassifier(**params)
            m.fit(X_ftr[itr_idx], y_ftr.iloc[itr_idx])
            kappas.append(cohen_kappa_score(y_ftr.iloc[ival_idx], m.predict(X_ftr[ival_idx]), weights='quadratic'))
        return np.mean(kappas)

    study_et = optuna.create_study(direction='maximize')
    study_et.optimize(objective_et, n_trials=N_INNER_TRIALS)
    best_et = {**study_et.best_params, 'random_state': SEED, 'n_jobs': -1}
    mdl_et = ExtraTreesClassifier(**best_et)
    mdl_et.fit(X_ftr, y_ftr)
    oof_preds['extratrees'][outer_val_idx] = mdl_et.predict_proba(X_fval)
    test_preds['extratrees'] += mdl_et.predict_proba(X_test.values) / OUTER_FOLDS
    print(f"  extratrees kappa fold: {cohen_kappa_score(y_fval, mdl_et.predict(X_fval), weights='quadratic'):.4f}")

print("\n=== OOF Kappa por modelo ===")
for name in model_names:
    k = cohen_kappa_score(y_train, oof_preds[name].argmax(axis=1), weights='quadratic')
    print(f"  {name}: {k:.4f}")



=== Fold externo 1/5 ===
  lgbm       kappa fold: 0.3989
  catboost   kappa fold: 0.3973
  rf         kappa fold: 0.3713
  extratrees kappa fold: 0.3416

=== Fold externo 2/5 ===
  lgbm       kappa fold: 0.4136
  catboost   kappa fold: 0.3563
  rf         kappa fold: 0.3528
  extratrees kappa fold: 0.3274

=== Fold externo 3/5 ===
  lgbm       kappa fold: 0.3940
  catboost   kappa fold: 0.3480
  rf         kappa fold: 0.3602
  extratrees kappa fold: 0.3752

=== Fold externo 4/5 ===
  lgbm       kappa fold: 0.3636
  catboost   kappa fold: 0.3877
  rf         kappa fold: 0.3425
  extratrees kappa fold: 0.3456

=== Fold externo 5/5 ===
  lgbm       kappa fold: 0.3982
  catboost   kappa fold: 0.3817
  rf         kappa fold: 0.3347
  extratrees kappa fold: 0.3359

=== OOF Kappa por modelo ===
  lgbm: 0.3940
  catboost: 0.3742
  rf: 0.3522
  extratrees: 0.3452


Construcción de meta-features

In [4]:
meta_train = np.hstack([oof_preds[name]  for name in model_names])
meta_test  = np.hstack([test_preds[name] for name in model_names])

print(f"Meta-features train: {meta_train.shape}")  # 4 modelos x 5 clases = 20
print(f"Meta-features test:  {meta_test.shape}")


Meta-features train: (11994, 20)
Meta-features test:  (2999, 20)


 Meta-modelo

In [5]:
meta_model = LogisticRegression(max_iter=1000, random_state=SEED)
meta_model.fit(meta_train, y_train)

y_pred_oof   = meta_model.predict(meta_train)
y_pred_final = meta_model.predict(meta_test)

kappa_oof = cohen_kappa_score(y_train, y_pred_oof, weights='quadratic')
print(f"Kappa OOF meta-modelo: {kappa_oof:.4f}")

if y_test is not None:
    kappa_test = cohen_kappa_score(y_test, y_pred_final, weights="quadratic")
    print(f"Kappa TEST  meta-modelo: {kappa_test:.4f}")


Kappa OOF meta-modelo: 0.4120


### Meta-modelos adicionales: Ridge y LGBM ligero

In [6]:
from sklearn.linear_model import RidgeClassifierCV

# ── Ridge (selecciona alpha automáticamente via CV interno) ────────────────────
ridge_meta = RidgeClassifierCV(alphas=[0.01, 0.1, 1, 10, 100], cv=5)
ridge_meta.fit(meta_train, y_train)

y_pred_ridge_oof   = ridge_meta.predict(meta_train)
y_pred_ridge_final = ridge_meta.predict(meta_test)

kappa_ridge_oof  = cohen_kappa_score(y_train, y_pred_ridge_oof,   weights='quadratic')
kappa_ridge_test = cohen_kappa_score(y_test,  y_pred_ridge_final, weights='quadratic')
print(f"Ridge    | alpha={ridge_meta.alpha_:.4f} | OOF: {kappa_ridge_oof:.4f} | TEST: {kappa_ridge_test:.4f}")

# ── LGBM ligero ────────────────────────────────────────────────────────────────
lgbm_meta = LGBMClassifier(
    n_estimators=100, max_depth=3, num_leaves=15,
    learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=1.0,
    class_weight='balanced', random_state=SEED, verbosity=-1,
)
lgbm_meta.fit(meta_train, y_train)

y_pred_lgbm_oof   = lgbm_meta.predict(meta_train)
y_pred_lgbm_final = lgbm_meta.predict(meta_test)

kappa_lgbm_oof  = cohen_kappa_score(y_train, y_pred_lgbm_oof,   weights='quadratic')
kappa_lgbm_test = cohen_kappa_score(y_test,  y_pred_lgbm_final, weights='quadratic')
print(f"LGBM     |             | OOF: {kappa_lgbm_oof:.4f} | TEST: {kappa_lgbm_test:.4f}")

# ── Comparación de los tres meta-modelos ───────────────────────────────────────
print(f"\nResumen meta-modelos:")
print(f"  LogisticRegression | OOF: {kappa_oof:.4f}  | TEST: {kappa_test:.4f}")
print(f"  Ridge              | OOF: {kappa_ridge_oof:.4f}  | TEST: {kappa_ridge_test:.4f}")
print(f"  LGBM ligero        | OOF: {kappa_lgbm_oof:.4f}  | TEST: {kappa_lgbm_test:.4f}")


InvalidParameterError: The 'y1' parameter of cohen_kappa_score must be an array-like. Got None instead.

In [8]:
from collections import Counter

print("Kappa individual en TEST:")
for name in model_names:
    base_preds = test_preds[name].argmax(axis=1)
    k = cohen_kappa_score(y_test, base_preds, weights='quadratic')
    print(f"  {name}: {k:.4f}  | pred dist: {dict(sorted(Counter(base_preds).items()))}")


Kappa individual en TEST:
  lgbm: 0.0571  | pred dist: {0: 33, 1: 2396, 2: 42, 3: 167, 4: 361}
  catboost: 0.1016  | pred dist: {0: 3, 1: 2331, 2: 85, 3: 75, 4: 505}
  rf: 0.2179  | pred dist: {0: 31, 1: 946, 2: 292, 3: 145, 4: 1585}
  extratrees: 0.1530  | pred dist: {0: 3, 1: 1552, 2: 139, 3: 139, 4: 1166}


In [11]:
print("=" * 55)
print("DIFERENCIA DE MEDIA TRAIN vs TEST (top 10)")
print("=" * 55)
diff = (X_train.mean() - X_test.mean()).abs().sort_values(ascending=False)
print(diff.head(10))

print("\n" + "=" * 55)
print("STATE")
print("=" * 55)
print(f"Train - únicos: {X_train['State'].nunique()} | min: {X_train['State'].min():.0f} | max: {X_train['State'].max():.0f} | mean: {X_train['State'].mean():.2f}")
print(f"Test  - únicos: {X_test['State'].nunique()}  | min: {X_test['State'].min():.0f} | max: {X_test['State'].max():.0f} | mean: {X_test['State'].mean():.2f}")
print(f"\nTrain value_counts:\n{X_train['State'].value_counts().head(10)}")
print(f"\nTest  value_counts:\n{X_test['State'].value_counts().head(10)}")

print("\n" + "=" * 55)
print("BREED1")
print("=" * 55)
print(f"Train - únicos: {X_train['Breed1'].nunique()} | min: {X_train['Breed1'].min():.0f} | max: {X_train['Breed1'].max():.0f} | mean: {X_train['Breed1'].mean():.2f}")
print(f"Test  - únicos: {X_test['Breed1'].nunique()}  | min: {X_test['Breed1'].min():.0f} | max: {X_test['Breed1'].max():.0f} | mean: {X_test['Breed1'].mean():.2f}")

print("\n" + "=" * 55)
print("RESCUER_LISTINGS")
print("=" * 55)
if 'rescuer_listings' in X_train.columns:
    print(f"Train - únicos: {X_train['rescuer_listings'].nunique()} | min: {X_train['rescuer_listings'].min()} | max: {X_train['rescuer_listings'].max()} | mean: {X_train['rescuer_listings'].mean():.2f}")
    print(f"Test  - únicos: {X_test['rescuer_listings'].nunique()}  | min: {X_test['rescuer_listings'].min()} | max: {X_test['rescuer_listings'].max()} | mean: {X_test['rescuer_listings'].mean():.2f}")
    print(f"\nTrain value_counts (top 10):\n{X_train['rescuer_listings'].value_counts().head(10)}")
    print(f"\nTest  value_counts (top 10):\n{X_test['rescuer_listings'].value_counts().head(10)}")
else:
    print("rescuer_listings no está en las features")


DIFERENCIA DE MEDIA TRAIN vs TEST (top 10)
State           41341.575789
Breed1            156.736509
Breed2             47.627618
Type                1.012762
MaturitySize        1.003274
Dewormed            1.003075
Health              1.001328
Sterilized          0.998847
Gender              0.993868
FurLength           0.993336
dtype: float64

STATE
Train - únicos: 14 | min: 0 | max: 13 | mean: 4.92
Test  - únicos: 14  | min: 41324 | max: 41415 | mean: 41346.49

Train value_counts:
State
2     6991
12    3059
3      680
7      397
4      325
5      197
0      113
1       95
6       71
10      22
Name: count, dtype: int64

Test  value_counts:
State
41326    1723
41401     786
41327     163
41336     110
41330      95
41332      56
41324      24
41325      15
41335      14
41361       4
Name: count, dtype: int64

BREED1
Train - únicos: 166 | min: 0 | max: 165 | mean: 108.92
Test  - únicos: 110  | min: 0 | max: 307 | mean: 265.65

RESCUER_LISTINGS
Train - únicos: 1 | min: 1 | max: 1 | 